# Notebook 2: Fine-Tune a Summarization Model

Fine-tune a pre-trained summarization model on cybersecurity news data.

**Steps:**
1. Prepare training data
2. Configure training parameters
3. Fine-tune with HuggingFace Trainer
4. Evaluate with ROUGE
5. Export to Google Drive for use in phase3

In [ ]:
!pip install transformers torch datasets accelerate rouge-score wandb -q

In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Configuration

Change the cell below to switch models or data sources.

In [ ]:
# ============================================
# CONFIGURATION - Edit this cell
# ============================================

# Base model to fine-tune
BASE_MODEL = "facebook/bart-large-cnn"  # Options: facebook/bart-large-cnn, sshleifer/distilbart-cnn-6-6, google/pegasus-cnn_dailymail

# Output model name
MODEL_NAME = "cyber-bart-summarizer"

# Data source options:
#   "sample" - use built-in sample data
#   "upload" - upload a JSONL file from local machine
#   "drive"  - load from Google Drive
#   "huggingface" - load a HuggingFace dataset
DATA_SOURCE = "sample"

# If DATA_SOURCE == "drive", set the path
DRIVE_DATA_PATH = "/content/drive/MyDrive/model-lab/data/train.jsonl"

# If DATA_SOURCE == "huggingface", set the dataset
HF_DATASET = "allenai/cord19"
HF_SPLIT = "train"

# Training parameters
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 3e-5
MAX_INPUT_LEN = 1024
MAX_TARGET_LEN = 150
WARMUP_STEPS = 500
GRADIENT_ACCUMULATION = 2
FP16 = True

OUTPUT_DIR = f"./results/{MODEL_NAME}"
print(f"Config: {BASE_MODEL} -> {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")

## Load Training Data

In [ ]:
SAMPLE_DATA = [
    {"article": "A critical vulnerability in Microsoft Exchange Server, tracked as CVE-2024-12345, is being actively exploited by a Chinese threat group dubbed Storm-0558. The flaw allows remote code execution without authentication, giving attackers full access to email accounts and administrative privileges. Microsoft released an out-of-band security update on Tuesday, urging all Exchange Server administrators to patch immediately. The company confirmed that at least 30 organizations in the United States have been compromised, including government agencies and defense contractors. CISA has added the vulnerability to its Known Exploited Vulnerabilities catalog, requiring federal agencies to patch within 21 days.",
     "summary": "Microsoft Exchange Server vulnerability CVE-2024-12345 is being actively exploited by Chinese threat group Storm-0558, allowing unauthenticated remote code execution. At least 30 US organizations have been compromised. Microsoft released an emergency patch and CISA added the flaw to its Known Exploited Vulnerabilities catalog."},
    {"article": "Ransomware attacks on healthcare organizations surged 94% in 2023 compared to the previous year, according to a new report from Sophos. The average ransom demand reached $1.5 million, while the average cost of recovery exceeded $3.4 million including downtime, legal fees, and regulatory fines. The report surveyed 500 healthcare IT professionals across 14 countries and found that 67% of organizations hit by ransomware paid the ransom, though only 58% of those recovered all their data. LockBit and BlackCat were the most prevalent ransomware families targeting healthcare.",
     "summary": "Ransomware attacks on healthcare rose 94% in 2023, with average ransom demands of $1.5M and recovery costs exceeding $3.4M. 67% of victims paid ransoms, but only 58% recovered all data. LockBit and BlackCat were the most common ransomware families targeting healthcare."},
    {"article": "The Cybersecurity and Infrastructure Security Agency published new binding operational directives requiring all federal civilian agencies to adopt multifactor authentication and patch known vulnerabilities within strict timelines. Directive BOD 23-02 mandates that agencies implement phishing-resistant MFA for all staff and contractor accounts within 180 days, while BOD 23-01 requires patching of critical vulnerabilities within 14 days and high-severity flaws within 30 days. Agencies that fail to comply must submit remediation plans to CISA.",
     "summary": "CISA issued binding operational directives requiring federal agencies to implement phishing-resistant MFA within 180 days and patch critical vulnerabilities within 14 days. Non-compliant agencies must submit remediation plans."},
    {"article": "Security researchers at CrowdStrike have identified a new supply chain attack targeting popular npm packages used by over 10 million developers. The attack involves malicious code injected into legitimate packages through compromised developer accounts. The malicious payloads establish reverse shells and exfiltrate environment variables containing API keys and database credentials. Affected packages include utility-lib, data-helpers, and secure-config. Developers are advised to update to the latest patched versions and rotate any exposed credentials immediately.",
     "summary": "CrowdStrike identified a supply chain attack targeting npm packages used by over 10 million developers. Malicious code injected through compromised accounts creates reverse shells and steals API keys. Developers should update affected packages and rotate exposed credentials."},
    {"article": "The LockBit ransomware group has claimed responsibility for the attack on Boeing, one of the world's largest aerospace and defense contractors. The threat actor added Boeing to its leak site on Friday, claiming to have exfiltrated a significant amount of data. Boeing confirmed it was aware of the incident and was investigating. Security analysts note that Boeing's potential exposure of sensitive defense-related data could have national security implications. The incident comes as LockBit has been increasingly targeting large enterprises and critical infrastructure organizations.",
     "summary": "LockBit ransomware group claimed responsibility for attacking Boeing, listing the aerospace giant on its leak site and claiming data exfiltration. Boeing confirmed it is investigating. Analysts warn the breach could have national security implications due to Boeing's defense contracts."},
    {"article": "Google's Threat Analysis Group revealed that government-backed hackers from Russia and China exploited zero-day vulnerabilities in Chrome and Windows to target journalists and policy experts. The campaigns used watering hole attacks on news websites and spear-phishing emails with malicious attachments. Google patched three Chrome vulnerabilities and Microsoft released emergency updates for two Windows flaws. The company also updated its Chrome sandbox and added new heuristics to detect suspicious download patterns.",
     "summary": "Government-backed hackers from Russia and China exploited zero-day vulnerabilities in Chrome and Windows to target journalists and policy experts. Google patched three Chrome flaws and Microsoft released emergency Windows updates. Attack methods included watering hole attacks and spear-phishing."},
    {"article": "Cloudflare reported mitigating the largest DDoS attack ever recorded, peaking at 71 million requests per second. The attack targeted a prominent Asian cryptocurrency exchange and lasted approximately 90 minutes. The volumetric attack leveraged a botnet of over 30,000 compromised IoT devices and cloud servers. Cloudflare's automated detection systems identified and blocked the malicious traffic within seconds. The company noted that DDoS attacks exceeding 10 million rps have increased 300% year over year.",
     "summary": "Cloudflare mitigated a record 71 million rps DDoS attack targeting an Asian cryptocurrency exchange. The 90-minute attack used a 30,000-device botnet of compromised IoT devices and cloud servers. DDoS attacks exceeding 10 million rps have increased 300% year over year."},
    {"article": "The European Union Agency for Cybersecurity released its annual threat report highlighting that supply chain compromises increased by 180% in 2023. The report identifiedsoftware supply chains as the primary attack vector, with threat actors increasingly targeting CI/CD pipelines and package repositories. State-sponsored groups accounted for approximately 40% of sophisticated attacks, with pro-Russia and pro-China groups being most active. The report recommends implementing SBOM requirements, adopting zero-trust architectures, and improving vendor security assessments.",
     "summary": "ENISA's annual threat report shows supply chain compromises increased 180% in 2023, with software supply chains as the primary vector targeting CI/CD pipelines. State-sponsored groups from Russia and China accounted for 40% of sophisticated attacks. Recommendations include SBOM requirements and zero-trust adoption."},
]

if DATA_SOURCE == "sample":
    train_data = SAMPLE_DATA
    print(f"Using sample data: {len(train_data)} articles")
    
elif DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    train_data = []
    with open(filename) as f:
        for line in f:
            if line.strip():
                train_data.append(json.loads(line))
    print(f"Loaded {len(train_data)} articles from {filename}")
    
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    train_data = []
    with open(DRIVE_DATA_PATH) as f:
        for line in f:
            if line.strip():
                train_data.append(json.loads(line))
    print(f"Loaded {len(train_data)} articles from Drive")
    
elif DATA_SOURCE == "huggingface":
    from datasets import load_dataset
    hf_data = load_dataset(HF_DATASET, split=HF_SPLIT)
    train_data = [{"article": r["article"], "summary": r["summary"]} for r in hf_data]
    print(f"Loaded {len(train_data)} articles from HuggingFace")

print(f"\nSample article (first 150 chars):\n{train_data[0]['article'][:150]}...")
print(f"\nSample summary:\n{train_data[0]['summary'][:150]}...")

## Tokenize & Prepare Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

dataset = Dataset.from_list(train_data)
split = dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

print(f"Train: {len(train_ds)} samples")
print(f"Eval: {len(eval_ds)} samples")

def preprocess(examples):
    inputs = tokenizer(
        examples["article"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    targets = tokenizer(
        examples["summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("Dataset prepared!")

## Train

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,
    fp16=FP16,
    logging_steps=10,
    warmup_steps=WARMUP_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")
trainer.train()
print("Training complete!")

## Evaluate with ROUGE

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, device=0 if DEVICE == "cuda" else -1)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

eval_samples = split["test"]
r1_list, r2_list, rl_list = [], [], []

for i in range(min(len(eval_samples), len(train_data))):
    article = eval_samples[i]["article"] if "article" in eval_samples.column_names else train_data[i]["article"]
    reference = eval_samples[i]["summary"] if "summary" in eval_samples.column_names else train_data[i]["summary"]
    
    try:
        result = summarizer(article, max_length=MAX_TARGET_LEN, min_length=30, do_sample=False)
        prediction = result[0]["summary_text"]
    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    
    scores = scorer.score(reference, prediction)
    r1_list.append(scores["rouge1"].fmeasure)
    r2_list.append(scores["rouge2"].fmeasure)
    rl_list.append(scores["rougeL"].fmeasure)

print(f"\n=== ROUGE Scores (Fine-tuned {MODEL_NAME}) ===")
print(f"ROUGE-1: {sum(r1_list)/len(r1_list):.4f}")
print(f"ROUGE-2: {sum(r2_list)/len(r2_list):.4f}")
print(f"ROUGE-L: {sum(rl_list)/len(rl_list):.4f}")

## Show Sample Summaries

In [ ]:
test_articles = [
    "The FBI and CISA jointly issued an advisory warning that the Royal ransomware group is actively targeting healthcare and public health organizations. The advisory describes Royal as a threat actor that has compromised at least 11 organizations since September 2022, using callback phishing and vulnerability exploitation as initial access vectors. Once inside, Royal operators exfiltrate data and deploy ransomware with encryption that cannot be decrypted without the private key. The advisory recommends implementing network segmentation, maintaining offline backups, and training staff on phishing awareness.",
    "OpenAI disclosed that its AI systems were targeted by an adversarial attack aimed at extracting sensitive training data. Researchers found that carefully crafted prompts could cause the model to regurgitate memorized text from its training corpus, including personal information and copyrighted content. OpenAI has since implemented new mitigation strategies including differential privacy techniques and improved data sanitization pipelines. The company also published a bug bounty program rewarding researchers up to $20,000 for responsible disclosure of model vulnerabilities.",
]

for article in test_articles:
    print(f"\nARTICLE: {article[:100]}...")
    result = summarizer(article, max_length=150, min_length=30, do_sample=False)
    print(f"SUMMARY: {result[0]['summary_text']}")
    print("-" * 40)

## Save Model

In [ ]:
# Save locally
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

drive_dir = Path(f'/content/drive/MyDrive/model-lab/models/{MODEL_NAME}')
drive_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(drive_dir))
tokenizer.save_pretrained(str(drive_dir))

# Save training config
config = {
    "base_model": BASE_MODEL,
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "train_samples": len(train_ds),
    "eval_samples": len(eval_ds),
}
with open(drive_dir / "training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nModel saved to Google Drive: {drive_dir}")
print(f"\nTo use in phase3, copy the model to the phase3 directory and update cyber_summarizer.py:")
print(f"  model_path = \"{drive_dir}\"")
print(f"  Or push to HuggingFace Hub and use the repo ID as model_path.")

## (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment to push to HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub(f"your-username/{MODEL_NAME}")
# tokenizer.push_to_hub(f"your-username/{MODEL_NAME}")
# print(f"Pushed to: https://huggingface.co/your-username/{MODEL_NAME}")